In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
df_loc = df[
    (df["ISSUE"].str.strip().str.lower() == "location") &
    (df["Network ID"] != 11)
]
df_loc_new =  df_loc[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_loc_new


,Station ID,Unique Names,Unique Location (String),Native ID
0,12617.0,Martin's Gulch,"123.764438 W, 48.51616 N, Elev. 512.14 m -> 12...",FW007
1,12998.0,ALEX FRASER TOP,"122.94392 W, 49.16187 N, Elev. null m -> 122.9...",14096
2,12999.0,ALEX FRASER CROSS BEAM,"122.94391 W, 49.1618 N, Elev. null m -> 122.94...",14097
3,2733.0,Cayoosh Summit,"122.47403 W, 50.37978 N, Elev. 1350 m -> 122.4...",26224
4,12469.0,Big Bar,"123.700556 W, 48.516111 N, Elev. 180 m -> 121....",28071
...,...,...,...,...
180,2490.0,Parsnip Upper,"122.1458333 W, 54.6125 N, Elev. 790 m -> 122.1...",PAR
181,13124.0,Salmon River Below Campbell Lake Diversion 2,"125.669767 W, 50.093619 N, Elev. 226 m -> 125....",SBD2
182,12409.0,Townsend,"122.24 W, 56.83 N, Elev. 1011 m -> 122.23972 W...",TNS
183,13117.0,Tsaydaychi Lake,"124.096333 W, 55.41635 N, Elev. 1189 m -> 124....",TSY


In [5]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df_loc_new[cols] = df_loc_new['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)
# 
df_loc_new = df_loc_new.drop(columns=['Unique Location (String)'])

In [12]:
df_loc_new[150:200]

,Station ID,Unique Names,Native ID,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev
150,2349.0,SKOONKA,1024,50.438690,-121.551500,1557.0,50.438700,-121.551500,1557.0
151,2351.0,ZZ DANIELS PASS,1027,50.402639,-124.412222,856.0,50.402600,-124.412200,856.0
152,2352.0,ZZ MONTROSE INTAKE,1028,50.656944,-124.114167,644.0,50.657200,-124.114300,644.0
153,2353.0,FIVE MILE,1029,50.910890,-122.688890,865.0,50.910890,-122.688890,875.0
154,2355.0,ZZ EAST TOBA,1032,50.647778,-123.883056,731.0,50.647900,-123.883200,731.0
155,2356.0,ZZ AGASSIZ GM,1037,49.271111,-121.780556,19.0,49.271300,-121.780800,19.0
156,2359.0,BIG SILVER 2,1040,49.691000,-121.859620,561.0,49.723400,-121.836600,570.0
157,2362.0,OSBORN,1045,56.558330,-120.381940,700.0,56.555300,-120.395200,700.0
158,2369.0,SPLINTLUM,1055,50.351300,-121.654950,424.0,50.351100,-121.654900,424.0
159,10983.0,KOOCANUSA,1075,49.046940,-115.225280,804.0,49.046940,-115.225300,804.0


In [7]:
check_sql = sa.text("""
SELECT
    station_id,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


In [8]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_loc_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  old_lat) and
            same(db_row.lon,  old_lon) and
            same(db_row.elev, old_elev)
        ):
            print(
                f"⚠️ Station {station_id} skipped (DB values differ)\n"
                f"   DB : lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: lat={old_lat}, lon={old_lon}, elev={old_elev}"
            )
            continue


⚠️ Station 2423 skipped (DB values differ)
   DB : lat=53.23395, lon=-126.1132667, elev=860
   XLS: lat=53.234, lon=-126.113267, elev=860.0


In [13]:
import numpy as np

update_sql = sa.text("""
UPDATE meta_history
SET
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_loc_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_lat}, {old_lon}, {old_elev}) → "
                f"({new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 12617: (48.51616, -123.764438, 512.14) → (48.51616, -123.764438, 512.0)
✅ Updated station 12998: (49.16187, -122.94392, nan) → (49.16187, -122.94392, 130.0)
✅ Updated station 12999: (49.1618, -122.94391, nan) → (49.1618, -122.94391, 80.0)
✅ Updated station 2733: (50.37978, -122.47403, 1350.0) → (50.37978, -122.47403, 1250.0)
✅ Updated station 12469: (48.516111, -123.700556, 180.0) → (51.14693, -121.50162, 1049.0)
✅ Updated station 12510: (48.516111, -123.700556, 180.0) → (51.1929, -121.5295, 1093.0)
✅ Updated station 2761: (49.75778, -119.12639, 1220.0) → (49.75778, -119.12639, 1200.0)
✅ Updated station 2771: (50.04631, -117.17914, 1080.0) → (50.04631, -117.17914, 1070.0)
✅ Updated station 13027: (49.37254, -115.04689, nan) → (49.37254, -115.04689, 1998.0)
✅ Updated station 2800: (51.27303, -116.76139, 1140.0) → (51.27483, -116.76653, 1140.0)
✅ Updated station 12988: (51.02732, -116.53516, nan) → (51.02732, -116.53516, 797.0)
✅ Updated station 12997: (50.43048, -116.3

In [14]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_loc_new.iterrows():

        station_id = int(row["Station ID"])

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  new_lat) and
            same(db_row.lon,  new_lon) and
            same(db_row.elev, new_elev)
        ):
            print(
                f"⚠️ Station {station_id} skipped (DB values differ)\n"
                f"   DB : lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: lat={new_lat}, lon={new_lon}, elev={new_elev}"
            )
        else:
            print(
                f"✅ Station {station_id} already matches new values "
                f"(lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev})"
            )


✅ Station 12617 already matches new values (lat=48.51616, lon=-123.764438, elev=512.0)
✅ Station 12998 already matches new values (lat=49.16187, lon=-122.94392, elev=130.0)
✅ Station 12999 already matches new values (lat=49.1618, lon=-122.94391, elev=80.0)
✅ Station 2733 already matches new values (lat=50.37978, lon=-122.47403, elev=1250.0)
✅ Station 12469 already matches new values (lat=51.14693, lon=-121.50162, elev=1049.0)
✅ Station 12510 already matches new values (lat=51.1929, lon=-121.5295, elev=1093.0)
✅ Station 2761 already matches new values (lat=49.75778, lon=-119.12639, elev=1200.0)
✅ Station 2771 already matches new values (lat=50.04631, lon=-117.17914, elev=1070.0)
✅ Station 13027 already matches new values (lat=49.37254, lon=-115.04689, elev=1998.0)
✅ Station 2800 already matches new values (lat=51.27483, lon=-116.76653, elev=1140.0)
✅ Station 12988 already matches new values (lat=51.02732, lon=-116.53516, elev=797.0)
✅ Station 12997 already matches new values (lat=50.430